# Reproducing the computational results

Run top to bottom. Paths resolve relative to this notebook, so nothing needs
editing. Every solve uses `glpk_exact`; see the deposit README for why the
default GLPK interface must not be used here.

The argument runs in five steps:

| | | |
|---|---|---|
| 1 | CHO-K1, scenario 52 | coupling |
| 2 | CHO-K1, BCAT1 knockout with mild hypothermia in place of the growth regulations | coupling |
| 3 | CHO-K1 with BCAT2 added, scenario 52 | coupling lost |
| 4 | iCHO3K, scenario 52 | coupling lost |
| 5 | iCHO3K, scenario 52 with BCAT2 and the leucine sinks knocked out | coupling |

Throughout, coupling means the **minimum** feasible IgG flux at a fixed growth
rate is above zero, so production is forced rather than merely permitted. A
single FBA solution would not show this.


## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import scipy.io as sio
from cobra import Reaction, Metabolite
from cobra.io import load_matlab_model
from cobra.flux_analysis import flux_variability_analysis as fva
import subprocess, sys

# Exact arithmetic throughout. Plain GLPK returns solutions that violate the IgG
# demand upper bound once the design regulations are applied - see the README.
SOLVER = "glpk_exact"
QCONV  = 0.315 * 24 * 150000          # flux -> pg/cell/day
IGG, BIO = "DM_igg[g]", "biomass_cho_producing"
DEP = Path.cwd().parent
SETS = {"D1": "FVA_EX_SUE_K1_NoEPO.mat",         "D2": "FVA_EX_SAR_FeedC_K1_NoEPO.mat",
        "D3": "FVA_EX_SAR_FeedAll_K1_NoEPO.mat", "D4": "FVA_EX_SAR_FeedAll40_K1_NoEPO.mat"}

K1 = load_matlab_model(str(DEP/"models"/"iCHO_K1_used.mat")); K1.solver = SOLVER
I3 = load_matlab_model(str(DEP/"models"/"iCHO3K.mat"));       I3.solver = SOLVER
REF = [r.id for r in K1.reactions]
BND = {d: sio.loadmat(str(DEP/"constraints"/f), squeeze_me=True)["FVA_EX"]
       for d, f in SETS.items()}
UPTAKE = {d: abs(float(BND[d][REF.index("EX_leu_L_e_"), 1])) for d in SETS}

def k1_setup(m, ds, block_cho=True):
    """Dataset bounds on every reaction; objectives reopened."""
    for i, r in enumerate(m.reactions):
        r.bounds = (float(BND[ds][i, 0]), float(BND[ds][i, 1]))
    m.reactions.get_by_id(IGG).bounds = (0., 100.)
    m.reactions.get_by_id(BIO).bounds = (0., 100.)
    if block_cho:
        m.reactions.get_by_id("biomass_cho").bounds = (0., 0.)
    m.objective = m.reactions.get_by_id(BIO)

def scenario52(m, ds):
    """The seven regulations, RKO_Results.m line 712, e_up = 0.60.

    RTOTAL2t_b and RTOTAL3t_b are the export directions. In the irreversible
    model the upregulation raises the lower bound of the _b split to 60% of its
    range; on this reversible model that is an upper bound of ub + 0.60*lb.
    """
    for rid in ("THRA", "AACOAT", "LEUTA", "PSP_L"):
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    r = m.reactions.get_by_id("r1109")
    if r.lower_bound < 0:
        r.lower_bound = 0.                       # r1109_b knockout
    else:
        print("   note: r1109 is irreversible here, so the r1109_b knockout is inert")
    for rid in ("RTOTAL2t", "RTOTAL3t"):
        i = REF.index(rid)
        lb, ub = float(BND[ds][i, 0]), float(BND[ds][i, 1])
        m.reactions.get_by_id(rid).bounds = (lb, ub + 0.60 * lb)

def forced(m, bio_id, igg_id, mu, label):
    """Minimum IgG at fixed growth.

    The minimum, not a single solution: only a forced minimum above zero
    demonstrates coupling.
    """
    m.reactions.get_by_id(bio_id).bounds = (mu * (1 - 1e-9), mu)
    iv = fva(m, [m.reactions.get_by_id(igg_id)], fraction_of_optimum=0., processes=1)
    lo, hi = float(iv.minimum.iloc[0]), float(iv.maximum.iloc[0])
    print(f"   {label:38s} mu {mu:.9f}   qmAb {lo*QCONV:9.4f}   FVA width {hi-lo:.1e}")
    return lo

def add_bcat2(m, capacity):
    """BCAT2 is absent from this reconstruction. Add mitochondrial leucine and
    the two reactions that restore the route, capped at `capacity`."""
    leu_m = Metabolite("leu_L[m]", compartment="m")
    t = Reaction("LEUt5m"); t.bounds = (-1000., 1000.)
    t.add_metabolites({m.metabolites.get_by_id("leu_L[c]"): -1, leu_m: 1})
    a = Reaction("LEUTAm"); a.bounds = (-capacity, capacity)
    a.add_metabolites({m.metabolites.get_by_id("akg[m]"): -1, leu_m: -1,
                       m.metabolites.get_by_id("4mop[m]"): 1,
                       m.metabolites.get_by_id("glu_L[m]"): 1})
    m.add_reactions([t, a])

print("CHO-K1 :", len(K1.reactions), "reactions,", len(K1.genes), "genes")
print("iCHO3K :", len(I3.reactions), "reactions,", len(I3.genes), "genes")
print("forced minimum leucine uptake:", {d: round(v, 6) for d, v in UPTAKE.items()})

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, im, l, m, n, r, x


CHO-K1 : 3952 reactions, 1202 genes
iCHO3K : 8368 reactions, 2929 genes
forced minimum leucine uptake: {'D1': 0.012905, 'D2': 0.008593, 'D3': 0.00603, 'D4': 0.010148}


## Step 1 - CHO-K1, scenario 52

The seven regulations of scenario 52 under dataset D4, reproducing the predicted
operating point. The FVA interval width shows the solution is unique, so the
numbers do not depend on which vertex the solver returns.


In [2]:
with K1 as m:
    k1_setup(m, "D4", block_cho=False)
    scenario52(m, "D4")
    forced(m, BIO, IGG, m.slim_optimize(), "scenario 52")
print("   published:                             mu 0.012369243   qmAb   43.6783")

   scenario 52                            mu 0.012369243   qmAb   43.6783   FVA width 9.8e-14
   published:                             mu 0.012369243   qmAb   43.6783


## Step 2 - CHO-K1, BCAT1 knockout with mild hypothermia

Scenario 52's five growth-suppressing regulations are replaced by a direct cap on
growth, which is what mild hypothermia does experimentally. The BCAT1 knockout is
kept. Wild type at the same cap produces no antibody; the knockout is forced to.


In [3]:
CAP = 0.0125
for label, ko in (("wild type", ()), ("BCAT1 knockout", ("LEUTA",))):
    with K1 as m:
        k1_setup(m, "D4")
        for rid in ko:
            m.reactions.get_by_id(rid).bounds = (0., 0.)
        forced(m, BIO, IGG, CAP, f"{label}, growth capped at {CAP}")

   wild type, growth capped at 0.0125     mu 0.012500000   qmAb    0.0000   FVA width 5.6e-05
   BCAT1 knockout, growth capped at 0.0125 mu 0.012500000   qmAb   42.7884   FVA width 1.9e-05


## Step 3 - CHO-K1 with BCAT2 added

BCAT2 is absent from this reconstruction, consistent with *Bcat2* not being
expressed in CHO. Adding it back gives the surplus leucine a route that bypasses
IgG, and the coupling is lost once its capacity reaches that surplus.


In [4]:
MU52 = 0.012369243
for frac in (0.0, 0.05, 0.10, 0.25, 0.50, 1.00):
    with K1 as m:
        k1_setup(m, "D4")
        scenario52(m, "D4")
        add_bcat2(m, frac * UPTAKE["D4"])
        forced(m, BIO, IGG, MU52, f"BCAT2 capacity {frac*100:5.1f}% of uptake")

surplus = UPTAKE["D4"] - 0.540177 * MU52
print(f"\n   leucine surplus at this growth rate: {surplus:.6f}"
      f"  = {100*surplus/UPTAKE['D4']:.1f}% of uptake")
print("   BCAT2 abolishes the coupling once it can carry that surplus.")
print("   Measured Bcat2 in IgG-expressing CHO-K1: 0.09 RPKM against Bcat1 31.26,")
print("   i.e. 0.29% of BCAT1, two orders of magnitude below this threshold.")

   BCAT2 capacity   0.0% of uptake        mu 0.012369243   qmAb   43.6783   FVA width 7.5e-13
   BCAT2 capacity   5.0% of uptake        mu 0.012369243   qmAb   37.2850   FVA width 6.4e-06
   BCAT2 capacity  10.0% of uptake        mu 0.012369243   qmAb   30.8917   FVA width 1.3e-05
   BCAT2 capacity  25.0% of uptake        mu 0.012369243   qmAb   11.7118   FVA width 3.2e-05
   BCAT2 capacity  50.0% of uptake        mu 0.012369243   qmAb    0.0000   FVA width 4.6e-05
   BCAT2 capacity 100.0% of uptake        mu 0.012369243   qmAb    0.0000   FVA width 5.4e-05

   leucine surplus at this growth rate: 0.003467  = 34.2% of uptake
   BCAT2 abolishes the coupling once it can carry that surplus.
   Measured Bcat2 in IgG-expressing CHO-K1: 0.09 RPKM against Bcat1 31.26,
   i.e. 0.29% of BCAT1, two orders of magnitude below this threshold.


## Step 4 - iCHO3K, scenario 52

The same seven regulations in a more recent reconstruction. The design does not
transfer: growth is not restricted and no antibody is forced at any growth rate.


In [5]:
I3BIO, I3IGG = "biomass_cho_prod", "DM_igg_g"

def i3_setup(m, ds):
    """2022 boundary bounds mapped onto iCHO3K by exchange identifier.

    358 of 389 map; the 31 that do not are listed in the deposit README.
    """
    have = {r.id for r in m.reactions}
    for j, rid in enumerate(REF):
        if rid.startswith("EX_") and rid.rstrip("_") in have:
            m.reactions.get_by_id(rid.rstrip("_")).bounds = (float(BND[ds][j, 0]),
                                                             float(BND[ds][j, 1]))
    for rid in ("biomass_cho", "biomass_cho_s"):
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    m.reactions.get_by_id(I3IGG).bounds = (0., 1000.)
    m.objective = m.reactions.get_by_id(I3BIO)

with I3 as m:
    i3_setup(m, "D4")
    scenario52(m, "D4")
    mumax = m.slim_optimize()
    print(f"   maximum growth under scenario 52: {mumax:.9f}")
    for f in (0.0, 0.5, 1.0):
        with m as mm:
            forced(mm, I3BIO, I3IGG, f * mumax, f"growth at {f*100:5.1f}% of maximum")

   note: r1109 is irreversible here, so the r1109_b knockout is inert
   maximum growth under scenario 52: 0.044050233
   growth at   0.0% of maximum            mu 0.000000000   qmAb    0.0000   FVA width 1.6e-04
   growth at  50.0% of maximum            mu 0.022025116   qmAb    0.0000   FVA width 1.1e-04
   growth at 100.0% of maximum            mu 0.044050233   qmAb    0.0000   FVA width 8.1e-13


## Step 5 - iCHO3K, scenario 52 with the additional knockouts

iCHO3K carries an active mitochondrial BCAT2 (`LEUTAm`) and three leucine sinks
left at default bounds: the Phe-Leu dipeptide export, apolipoprotein A1 assembly
and glycogenin assembly. Removing those four restores the coupling.


In [6]:
SINKS = ("EX_pheleu_e", "HMR_5166", "r1402")   # dipeptide, apoA1, glycogenin

with I3 as m:
    i3_setup(m, "D4")
    scenario52(m, "D4")
    for rid in ("LEUTAm",) + SINKS:            # BCAT2 and the three leucine sinks
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    mumax = m.slim_optimize()
    print(f"   maximum growth: {mumax:.9f}")
    for f in (0.0, 0.25, 0.5, 1.0):
        with m as mm:
            forced(mm, I3BIO, I3IGG, f * mumax, f"growth at {f*100:5.1f}% of maximum")

   note: r1109 is irreversible here, so the r1109_b knockout is inert
   maximum growth: 0.038931714
   growth at   0.0% of maximum            mu 0.000000000   qmAb  127.8662   FVA width 4.8e-05
   growth at  25.0% of maximum            mu 0.009732928   qmAb   73.6875   FVA width 7.2e-05
   growth at  50.0% of maximum            mu 0.019465857   qmAb   19.5088   FVA width 9.6e-05
   growth at 100.0% of maximum            mu 0.038931714   qmAb    0.0000   FVA width 6.6e-05


## Step 6 - why three leucine routes, and not fourteen

Fourteen plasma-protein assembly reactions in iCHO3K consume cytosolic leucine, so
flux variability alone would suggest fourteen escape routes. Eleven are futile
cycles: a paired degradation reaction returns exactly the leucine consumed, giving
no net drain. Only apolipoprotein A1 (secreted) and glycogenin (leucine retained in
the glycogen primer) are genuine sinks, alongside the Phe-Leu dipeptide export.
Adding each assembly reaction in turn to the double knockout confirms that only
`r1402` changes the outcome.


In [7]:
# The fourteen plasma-protein assembly reactions that consume cytosolic leucine.
PROTEINS = ["HMR_5166","r1092","r1093","r1094","r1095","r1098","r1099","r1100",
            "r1101","r1103","r1104","r1105","r1112","r1402"]
# Each is paired with a degradation reaction that returns its amino acids.
DEGRADER = {"HMR_5166":"r0808","r1092":"r1333","r1093":"r1335","r1094":"r1337",
            "r1095":"r1341","r1098":"r1343","r1099":"r1344","r1100":"r1345",
            "r1101":"r1348","r1103":"r1352","r1104":"r1354","r1105":"r1356",
            "r1112":"r1346"}

leu_c = I3.metabolites.leu_L_c
print("net leucine per unit flux (assembly minus its degrader):")
for rid, deg in DEGRADER.items():
    a = -I3.reactions.get_by_id(rid).metabolites[leu_c]
    b = I3.reactions.get_by_id(deg).metabolites.get(leu_c, 0.0)
    print(f"   {rid:10s} consumes {a:6.1f}, {deg:7s} returns {b:6.1f}"
          f"  ->  net {a-b:6.1f}  {'DRAIN' if abs(a-b) > 1e-9 else 'futile cycle'}")

# r1402 (glycogenin) has no degrader: leucine enters the glycogen primer and stays.
print("\nadding one assembly reaction at a time to BCAT1+BCAT2 KO + the two exports:")
base_ko = ("LEUTA", "LEUTAm", "EX_pheleu_e", "EX_HC00004_e")
with I3 as m:
    i3_setup(m, "D4")
    for rid in base_ko: m.reactions.get_by_id(rid).bounds = (0., 0.)
    print(f"   {'(none)':12s} min IgG at mu = 0: {forced(m, I3BIO, I3IGG, 0.0, 'two exports only'):.4e}")
for rid in PROTEINS:
    if rid == "HMR_5166": continue
    with I3 as m:
        i3_setup(m, "D4")
        for r2 in base_ko + (rid,): m.reactions.get_by_id(r2).bounds = (0., 0.)
        m.reactions.get_by_id(I3BIO).bounds = (0., 0.)
        m.objective = m.reactions.get_by_id(I3IGG); m.objective.direction = "min"
        v = m.slim_optimize()
        flag = "   <-- restores coupling" if (v or 0) > 1e-12 else ""
        print(f"   + {rid:10s} min IgG at mu = 0: {v:.4e}{flag}")

net leucine per unit flux (assembly minus its degrader):
   HMR_5166   consumes   41.0, r0808   returns    0.0  ->  net   41.0  DRAIN
   r1092      consumes   64.0, r1333   returns   64.0  ->  net    0.0  futile cycle
   r1093      consumes   59.0, r1335   returns   59.0  ->  net    0.0  futile cycle
   r1094      consumes   51.0, r1337   returns   51.0  ->  net    0.0  futile cycle
   r1095      consumes  533.0, r1341   returns  533.0  ->  net    0.0  futile cycle
   r1098      consumes   12.0, r1343   returns   12.0  ->  net    0.0  futile cycle
   r1099      consumes   15.0, r1344   returns   15.0  ->  net    0.0  futile cycle
   r1100      consumes   11.0, r1345   returns   11.0  ->  net    0.0  futile cycle
   r1101      consumes  166.0, r1348   returns  166.0  ->  net    0.0  futile cycle
   r1103      consumes   48.0, r1352   returns   48.0  ->  net    0.0  futile cycle
   r1104      consumes   51.0, r1354   returns   51.0  ->  net    0.0  futile cycle
   r1105      consumes   6

## Step 7 - how much capacity each route needs to break the coupling

From the fully constrained baseline, each route is reopened to a capacity expressed
as leucine drain (leucine coefficient x flux) as a percentage of the dataset's forced
minimum leucine uptake. Forced IgG falls in proportion to the capacity granted and
survives to 99%. The growth rate below which IgG is forced is then found by bisection.


In [8]:
SINKS = {"EX_pheleu_e": 1.0, "HMR_5166": 41.0, "r1402": 39.0}   # leucine per unit flux

def i3_closed(m, ds):
    """BCAT1, BCAT2 and all three leucine sinks closed."""
    i3_setup(m, ds)
    for rid in ("LEUTA", "LEUTAm") + tuple(SINKS): m.reactions.get_by_id(rid).bounds = (0., 0.)

for ds in ("D4", "D2"):
    lm = UPTAKE[ds]
    print(f"--- {ds}: forced minimum leucine uptake {lm:.6f}")
    for rid, per in SINKS.items():
        row = []
        for frac in (0.50, 0.95, 0.99, 1.00):
            with I3 as m:
                i3_closed(m, ds)
                m.reactions.get_by_id(rid).bounds = (0., frac * lm / per)
                m.reactions.get_by_id(I3BIO).bounds = (0., 0.)
                m.objective = m.reactions.get_by_id(I3IGG); m.objective.direction = "min"
                v = m.slim_optimize() or 0.0
            row.append(f"{frac*100:5.0f}%: {v*QCONV:8.3f}")
        print(f"   {rid:12s} forced qmAb at mu = 0, by capacity granted   " + "  ".join(row))

    # growth rate below which IgG is forced, all three closed
    with I3 as m:
        i3_closed(m, ds)
        mumax = m.slim_optimize()
        lo, hi = 0.0, mumax
        for _ in range(30):
            mid = (lo + hi) / 2
            with m as mm:
                mm.reactions.get_by_id(I3BIO).bounds = (mid, mid)
                mm.objective = mm.reactions.get_by_id(I3IGG); mm.objective.direction = "min"
                v = mm.slim_optimize()
            if (v or 0) > 1e-12: lo = mid
            else: hi = mid
    print(f"   IgG forced below mu = {lo:.6f}  (max growth {mumax:.6f})\n")

--- D4: forced minimum leucine uptake 0.010148
   EX_pheleu_e  forced qmAb at mu = 0, by capacity granted      50%:   63.933     95%:    6.393     99%:    1.279    100%:    0.000
   HMR_5166     forced qmAb at mu = 0, by capacity granted      50%:   63.933     95%:    6.393     99%:    1.279    100%:    0.000
   r1402        forced qmAb at mu = 0, by capacity granted      50%:   63.933     95%:    6.393     99%:    1.279    100%:    0.000
   IgG forced below mu = 0.022971  (max growth 0.038932)

--- D2: forced minimum leucine uptake 0.008593
   EX_pheleu_e  forced qmAb at mu = 0, by capacity granted      50%:   54.134     95%:    5.413     99%:    1.083    100%:    0.000
   HMR_5166     forced qmAb at mu = 0, by capacity granted      50%:   54.134     95%:    5.413     99%:    1.083    100%:    0.000
   r1402        forced qmAb at mu = 0, by capacity granted      50%:   54.134     95%:    5.413     99%:    1.083    100%:    0.000
   IgG forced below mu = 0.019450  (max growth 0.038932)

## Step 8 - CHO-K1 forcing thresholds, and the iCHO3K bound mapping

With BCAT1 knocked out, IgG becomes obligatory below the growth rate at which biomass
can no longer absorb the forced leucine uptake. The bisected threshold is compared
with the closed form, uptake divided by the leucine coefficient of biomass. The last
block reports how many CHO-K1 exchange bounds have a counterpart in iCHO3K.


In [9]:
leu_coeff = abs(K1.reactions.get_by_id(BIO).metabolites[K1.metabolites.get_by_id("leu_L[c]")])
print(f"leucine coefficient of {BIO}: {leu_coeff:.6f}\n")

for ds in ("D2", "D4"):
    with K1 as m:
        k1_setup(m, ds)                      # blocks biomass_cho
        m.reactions.get_by_id("LEUTA").bounds = (0., 0.)
        mumax = m.slim_optimize()
        lo, hi = 0.0, mumax
        for _ in range(40):
            mid = (lo + hi) / 2
            with m as mm:
                mm.reactions.get_by_id(BIO).bounds = (mid, mid)
                mm.objective = mm.reactions.get_by_id(IGG); mm.objective.direction = "min"
                v = mm.slim_optimize()
            if (v or 0) > 1e-12: lo = mid
            else: hi = mid
    print(f"{ds}: forced leucine uptake {UPTAKE[ds]:.6f} | IgG forced below mu = {lo:.6f}"
          f" | uptake/coefficient = {UPTAKE[ds]/leu_coeff:.6f}")

# How many of the CHO-K1 exchange bounds have a counterpart in iCHO3K.
have = {r.id for r in I3.reactions}
bnds = {rid.rstrip("_") for rid in REF if rid.startswith("EX_")}
unmapped = sorted(k for k in bnds if k not in have)
print(f"\nCHO-K1 exchanges {len(bnds)}; mapped onto iCHO3K {len(bnds)-len(unmapped)}; "
      f"unmapped {len(unmapped)}:")
print("   " + ", ".join(unmapped))

leucine coefficient of biomass_cho_producing: 0.540177

D2: forced leucine uptake 0.008593 | IgG forced below mu = 0.015907 | uptake/coefficient = 0.015907
D4: forced leucine uptake 0.010148 | IgG forced below mu = 0.018787 | uptake/coefficient = 0.018787

CHO-K1 exchanges 389; mapped onto iCHO3K 358; unmapped 31:
   EX_C05965_e, EX_CLPND_e, EX_HC00229_e, EX_HC00955_e, EX_HC01577_e, EX_HC01610_e, EX_HC02176_e, EX_HC02203_e, EX_HC02204_e, EX_HC02205_e, EX_HC02207_e, EX_ak2lgchol_cho_e, EX_avite1_e, EX_avite2_e, EX_crm_cho_e, EX_crmp_cho_e, EX_dag_cho_e, EX_galside_cho_e, EX_ivcrn, EX_lpchol_cho_e, EX_mag_cho_e, EX_paf_cho_e, EX_pchol_cho_e, EX_pe_cho_e, EX_pglyc_cho_e, EX_ps_cho_e, EX_spc_cho_e, EX_tag_cho_e, EX_xolest2_cho_e, EX_xolest_cho_e, EX_yvite_e
